In [1]:
# torch, faiss (k-NN/perplexity graph construction) and numba-based internals (UMAP/PHATE) all
# get loaded into one process here; with unconstrained thread pools that combination makes macOS
# abort with a duplicate-OpenMP-runtime error (kernel crash / segfault). Capping thread counts
# before any of those libraries import, plus the (unsafe but standard) escape hatch as a second
# line of defense, avoids it (same fix as in rebuttal_distance_presevation_super_resolution.ipynb
# and rebuttal_query_interpolation_baselines.ipynb).
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')

import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess


def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

using Device: mps
Using device: mps


In [2]:
# Trained weights (ERA5/trained_models/regular/regular_papermodel2_2026-05-04_t12-01-43.pth)
# are not available on this machine -- only the cached outputs from a prior run/machine remain
# in this notebook. Per instruction, skipping NOFE for this run: reconstructing `config` by hand
# from the training/model/dataset config JSONs that produced this model id (same values that
# would have been in the missing per-run logged config), so PHATE/spatialPCA/Concord/fPCA (and
# the existing PCA/t-SNE/UMAP) can still be computed and evaluated.
model = None
config = {
    # from ERA5/configurations/regular/dataset_configs/dataset_reg_fixcrop_median_woPos_tkernel.json
    'lat_min': 30, 'lat_max': 75, 'lon_min': -12, 'lon_max': 33,
    'N_min': 500, 'N_max': 10000,
    'k': 10, 'perplexity': 10, 'sampling_method': 'random',
    'train_date_start_end': ['2018-01-01', '2018-12-31'],
    'test_date_start_end': ['2019-01-01', '2019-01-31'],
    'train_samples_per_date': 10, 'test_samples_per_date': 5,
    'feature_mode': 'channel_only', 'sigma_method': 'global_median',
    'vars': ['w', 'v', 'z', 'q', 'r', 't', 'd', 'o3', 'pv', 'u', 'vo'],
    # from ERA5/configurations/regular/model_configs/papermodel2.json (unused without the model)
    'width': 64, 'ker_width': 16, 'depth': 3, 'node_features': 11, 'edge_attr': 7, 'out_width': 3,
}

In [3]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [4]:
random.randint(100,1000)

650

In [5]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

bbox = lon_min, lat_min, lon_max, lat_max


print('generate data')

graphs=[]
locations_list=[]
X_list=[]

for date in dates:
    N=random.randint(1000,10000)
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method='ones')
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

generate data


total points available: 32761


total points available: 32761


total points available: 32761


total points available: 32761


total points available: 32761


... (22 outputs left out)

total points available: 32761


total points available: 32761


total points available: 32761


total points available: 32761


In [6]:
import umap

pred_nofe_list = []
pred_pca_list = []
pred_tsne_list = []
pred_umap_list = []

for sample in graphs:
    # NOFE -- skipped, trained weights are not available on this machine (see cell above)
    if model is not None:
        print('NOFE ...', end='\r')
        pred_nofe = model(sample).detach().cpu().numpy()
        pred_nofe_list.append(pred_nofe)
    
    X = sample.normed_values.detach().cpu().numpy()
    
    # PCA
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pred_pca_list.append(pred_pca)

    # t-SNE
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pred_tsne_list.append(pred_tsne)

    # UMAP -- umap-learn is installed directly in this env; run in-process rather than via the
    # separate "umap" conda env (that env does not exist on this machine).
    print('UMAP ...', end='\r')
    pred_umap = umap.UMAP(
        n_components=3, n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42
    ).fit_transform(X)
    pred_umap_list.append(pred_umap)


PCA ...


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normal

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


PCA ...


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/siche9897/miniconda3/envs/neural-oper

... (100 outputs left out)

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


PCA ...


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normal

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
# Additional baselines: PHATE, spatialPCA, Concord, functionalPCA -- same methods and
# implementations as rebuttal_distance_presevation_super_resolution.ipynb, applied here to
# the plain (non-cross, in-sample) graphs already built above.
import phate
import anndata as ad
import concord as ccd
import synthia as syn

spdir = f'{project_root}/ERA5/spatialpca'
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y


def concord_embed(X, n_components=3):
    adata = ad.AnnData(X=X.astype(np.float32))
    cur_ccd = ccd.Concord(
        adata=adata, input_feature=list(adata.var_names), device=device,
        latent_dim=n_components, verbose=False, preload_dense=True,
    )
    cur_ccd.fit_transform(output_key='Concord')
    return adata.obsm['Concord']


def fpca_embed(X, n_components=3):
    generator = syn.FPCADataGenerator()
    generator.fit(X.astype(np.float64), n_fpca_components=n_components)
    return generator.eig_scores


pred_phate_list = []
pred_spatialpca_list = []
pred_concord_list = []
pred_fpca_list = []

for i, sample in enumerate(graphs):
    print(f'{i}/{len(graphs)}')

    X = sample.normed_values.detach().cpu().numpy()
    coords = sample.normed_coords.detach().cpu().numpy()

    print('PHATE ...', end='\r')
    pred_phate = phate.PHATE(
        n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
    ).fit_transform(X)
    pred_phate_list.append(pred_phate)

    print('spatialPCA ...', end='\r')
    pred_spatialpca = spatialpca_embed(X, coords, spatialpca_call_id, n_components=3)
    pred_spatialpca_list.append(pred_spatialpca)

    print('Concord ...', end='\r')
    pred_concord = concord_embed(X, n_components=3)
    pred_concord_list.append(pred_concord)

    print('fPCA ...', end='\r')
    pred_fpca = fpca_embed(X, n_components=3)
    pred_fpca_list.append(pred_fpca)

spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)
print('\ndone')

spatialPCA server ready.
0/31


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: divide by zero encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: overflow encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:590: RuntimeWarning: invalid value encountered in matmul
  U = Q @ Uhat
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:20

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: divide by zero encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: overflow encountered in matmul
  return _core_matmul(x1, x2)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/numpy/linalg/_linalg.py:3383: RuntimeWarning: invalid value encountered in matmul
  return _core_matmul(x1, x2)


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/skle

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: 

/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: 

... (993 outputs left out)

Epoch 13 Training


Epoch 14 Training


/Users/siche9897/miniconda3/envs/neural-operators-new/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)



done


In [8]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X**2, axis=1, keepdims=True)  # (N, 1)
    
    dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
    
    # numerical stability: avoid tiny negatives
    dist2 = np.maximum(dist2, 0.0)
    
    return np.sqrt(dist2)

def distance_preservation(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # Correlation (1 = perfect preservation)
    corr = np.corrcoef(dH, dL)[0, 1]
    
    return corr

def stress(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # stress 
    stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))
    
    return stress

In [9]:

def edge_distances(X, edge_indices):
    # X: (N, d), edge_indices: (E, 2)
    diff = X[edge_indices[:, 0]] - X[edge_indices[:, 1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    # Correlation
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

#def weighted_edge_stress(H, L, edge_indices, weights):
#    dH = edge_distances(H, edge_indices)
#    dL = edge_distances(L, edge_indices)   
#    return np.sqrt(np.sum(weights * (dH - dL)**2) / np.sum(weights * dH**2))

In [10]:
if pred_nofe_list:  # skipped entirely this run -- NOFE weights unavailable, see cells above
    for i in range(len(graphs)):

        edge_indices = graphs[i].edge_index.detach().cpu().numpy()
        H = graphs[i].normed_values.detach().cpu().numpy()
        Lnofe = pred_nofe_list[i]
        Lpca = pred_pca_list[i]
        Ltsne = pred_tsne_list[i]
        Lumap = pred_umap_list[i]


        nofe_preservation = []
        nofe_stress = []
        nofe_edge_preservation = []
        nofe_edge_stress = []

        nofe_preservation.append(distance_preservation(H, Lnofe))
        nofe_stress.append(stress(H, Lnofe))
        nofe_edge_preservation.append(edge_distance_preservation(H, Lnofe, edge_indices))
        nofe_edge_stress.append(edge_stress(H, Lnofe, edge_indices))


        pca_preservation = []
        pca_stress = []
        pca_edge_preservation = []
        pca_edge_stress = []

        pca_preservation.append(distance_preservation(H, Lpca))
        pca_stress.append(stress(H, Lpca))
        pca_edge_preservation.append(edge_distance_preservation(H, Lpca, edge_indices))
        pca_edge_stress.append(edge_stress(H, Lpca, edge_indices))


        tsne_preservation = []
        tsne_stress = []
        tsne_edge_preservation = []
        tsne_edge_stress = []

        tsne_preservation.append(distance_preservation(H, Ltsne))
        tsne_stress.append(stress(H, Ltsne))
        tsne_edge_preservation.append(edge_distance_preservation(H, Ltsne, edge_indices))
        tsne_edge_stress.append(edge_stress(H, Ltsne, edge_indices))


        umap_preservation = []
        umap_stress = []
        umap_edge_preservation = []
        umap_edge_stress = []

        umap_preservation.append(distance_preservation(H, Lumap))
        umap_stress.append(distance_preservation(H, Lumap))
        umap_edge_preservation.append(edge_distance_preservation(H, Lumap, edge_indices))
        umap_edge_stress.append(edge_stress(H, Lumap, edge_indices))


    scores_nofe = [np.mean(nofe_preservation), np.mean(nofe_stress), np.mean(nofe_edge_preservation), np.mean(nofe_edge_stress)]
    scores_pca = [np.mean(pca_preservation), np.mean(pca_stress), np.mean(pca_edge_preservation), np.mean(pca_edge_stress)]
    scores_tsne = [np.mean(tsne_preservation), np.mean(tsne_stress), np.mean(tsne_edge_preservation), np.mean(tsne_edge_stress)]
    scores_umap = [np.mean(umap_preservation), np.mean(umap_stress), np.mean(umap_edge_preservation), np.mean(umap_edge_stress)]


    std_nofe = [np.mean(nofe_preservation), np.mean(nofe_stress), np.mean(nofe_edge_preservation), np.mean(nofe_edge_stress)]
    std_pca = [np.mean(pca_preservation), np.mean(pca_stress), np.mean(pca_edge_preservation), np.mean(pca_edge_stress)]
    std_tsne = [np.mean(tsne_preservation), np.mean(tsne_stress), np.mean(tsne_edge_preservation), np.mean(tsne_edge_stress)]
    std_umap = [np.mean(umap_preservation), np.mean(umap_stress), np.mean(umap_edge_preservation), np.mean(umap_edge_stress)]

    scores = pd.DataFrame({'nofe': scores_nofe,
                  'pca': scores_pca,
                  'tsne': scores_tsne,
                  'umap': scores_umap}, 
                 index=['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)'])
    #print(scores)
    print('nofe, pca, tsne, umap')
    for i, type in enumerate(['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)']):
        print(f'{type}: {scores_nofe[i]:.3f} $\\pm$ {std_nofe[i]:.3f} & {scores_pca[i]:.3f} $\\pm$ {std_pca[i]:.3f} & {scores_tsne[i]:.3f} $\\pm$ {std_tsne[i]:.3f} & {scores_umap[i]:.3f} $\\pm$ {std_umap[i]:.3f}')

In [11]:
nofe_preservation = []
nofe_stress = []
nofe_edge_preservation = []
nofe_edge_stress = []

pca_preservation = []
pca_stress = []
pca_edge_preservation = []
pca_edge_stress = []

tsne_preservation = []
tsne_stress = []
tsne_edge_preservation = []
tsne_edge_stress = []

umap_preservation = []
umap_stress = []
umap_edge_preservation = []
umap_edge_stress = []
    
for i in range(len(graphs)):

    edge_indices = graphs[i].edge_index.detach().cpu().numpy()
    H = graphs[i].normed_values.detach().cpu().numpy()
    Lpca = pred_pca_list[i]
    Ltsne = pred_tsne_list[i]
    Lumap = pred_umap_list[i]

    if pred_nofe_list:  # skipped this run -- NOFE weights unavailable, see cells above
        Lnofe = pred_nofe_list[i]
        nofe_preservation.append(distance_preservation(H, Lnofe))
        nofe_stress.append(stress(H, Lnofe))
        nofe_edge_preservation.append(edge_distance_preservation(H, Lnofe, edge_indices))
        nofe_edge_stress.append(edge_stress(H, Lnofe, edge_indices))

    pca_preservation.append(distance_preservation(H, Lpca))
    pca_stress.append(stress(H, Lpca))
    pca_edge_preservation.append(edge_distance_preservation(H, Lpca, edge_indices))
    pca_edge_stress.append(edge_stress(H, Lpca, edge_indices))

    tsne_preservation.append(distance_preservation(H, Ltsne))
    tsne_stress.append(stress(H, Ltsne))
    tsne_edge_preservation.append(edge_distance_preservation(H, Ltsne, edge_indices))
    tsne_edge_stress.append(edge_stress(H, Ltsne, edge_indices))

    umap_preservation.append(distance_preservation(H, Lumap))
    umap_stress.append(distance_preservation(H, Lumap))
    umap_edge_preservation.append(edge_distance_preservation(H, Lumap, edge_indices))
    umap_edge_stress.append(edge_stress(H, Lumap, edge_indices))


scores_pca = [np.mean(pca_preservation), np.mean(pca_stress), np.mean(pca_edge_preservation), np.mean(pca_edge_stress)]
scores_tsne = [np.mean(tsne_preservation), np.mean(tsne_stress), np.mean(tsne_edge_preservation), np.mean(tsne_edge_stress)]
scores_umap = [np.mean(umap_preservation), np.mean(umap_stress), np.mean(umap_edge_preservation), np.mean(umap_edge_stress)]

std_pca = [np.std(pca_preservation), np.std(pca_stress), np.std(pca_edge_preservation), np.std(pca_edge_stress)]
std_tsne = [np.std(tsne_preservation), np.std(tsne_stress), np.std(tsne_edge_preservation), np.std(tsne_edge_stress)]
std_umap = [np.std(umap_preservation), np.std(umap_stress), np.std(umap_edge_preservation), np.std(umap_edge_stress)]

score_cols = {'pca': scores_pca, 'tsne': scores_tsne, 'umap': scores_umap}
std_cols = {'pca': std_pca, 'tsne': std_tsne, 'umap': std_umap}
if pred_nofe_list:
    scores_nofe = [np.mean(nofe_preservation), np.mean(nofe_stress), np.mean(nofe_edge_preservation), np.mean(nofe_edge_stress)]
    std_nofe = [np.std(nofe_preservation), np.std(nofe_stress), np.std(nofe_edge_preservation), np.std(nofe_edge_stress)]
    score_cols = {'nofe': scores_nofe, **score_cols}
    std_cols = {'nofe': std_nofe, **std_cols}

scores = pd.DataFrame(score_cols, index=['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)'])
#print(scores)
print(', '.join(score_cols.keys()))
for i, type in enumerate(['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)']):
    print(f'{type}: ' + ' & '.join(f'{score_cols[name][i]:.3f} $\\pm$ {std_cols[name][i]:.3f}' for name in score_cols))

/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: divide by zero encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: overflow encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: invalid value encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)


pca, tsne, umap
corr (global): 0.891 $\pm$ 0.019 & 0.532 $\pm$ 0.066 & 0.468 $\pm$ 0.073
stress (global): 0.268 $\pm$ 0.014 & 4.366 $\pm$ 0.810 & 0.468 $\pm$ 0.073
corr (local): 1.000 $\pm$ 0.000 & 1.000 $\pm$ 0.000 & 1.000 $\pm$ 0.000
stress (local): 0.408 $\pm$ 0.171 & 0.649 $\pm$ 0.640 & 0.790 $\pm$ 0.207


In [12]:
# Comprehensive table: existing (pca/tsne/umap) + new (phate/spatialpca/concord/fpca)
# baselines, all computed the same way (correlation/stress over all pairs = "global",
# over graph edges = "local"). 'nofe' is excluded from this run -- trained weights aren't
# available on this machine (see cells above); pred_nofe_list is empty this run.
pred_lists = {
    'pca': pred_pca_list,
    'tsne': pred_tsne_list,
    'umap': pred_umap_list,
    'phate': pred_phate_list,
    'spatialpca': pred_spatialpca_list,
    'concord': pred_concord_list,
    'fpca': pred_fpca_list,
}
if pred_nofe_list:
    pred_lists = {'nofe': pred_nofe_list, **pred_lists}

def mean_std(arr):
    return f"{np.mean(arr):.4f} ± {np.std(arr):.4f}"

all_scores = {}
for name, preds in pred_lists.items():
    preservation, strs, edge_preservation, edge_strs = [], [], [], []
    for i in range(len(graphs)):
        edge_indices = graphs[i].edge_index.detach().cpu().numpy()
        H = graphs[i].normed_values.detach().cpu().numpy()
        L = preds[i]
        preservation.append(distance_preservation(H, L))
        strs.append(stress(H, L))
        edge_preservation.append(edge_distance_preservation(H, L, edge_indices))
        edge_strs.append(edge_stress(H, L, edge_indices))
    all_scores[name] = [
        mean_std(preservation), mean_std(strs), mean_std(edge_preservation), mean_std(edge_strs),
    ]

scores_all = pd.DataFrame(
    all_scores,
    index=['corr (global)', 'stress (global)', 'corr (local)', 'stress (local)'],
).T

pd.set_option('display.width', 120)
print(scores_all)

/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: divide by zero encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: overflow encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
/var/folders/px/747r5b694x947qk1vdh9b3th0000gp/T/ipykernel_33600/373983649.py:5: RuntimeWarning: invalid value encountered in matmul
  dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)


              corr (global)  stress (global)     corr (local)   stress (local)
pca         0.8908 ± 0.0191  0.2675 ± 0.0139  1.0000 ± 0.0000  0.4076 ± 0.1708
tsne        0.5318 ± 0.0664  4.3664 ± 0.8105  1.0000 ± 0.0000  0.6493 ± 0.6398
umap        0.4683 ± 0.0730  1.1110 ± 0.2820  1.0000 ± 0.0000  0.7900 ± 0.2065
phate       0.4805 ± 0.0826  0.9937 ± 0.0005  1.0000 ± 0.0000  0.9982 ± 0.0017
spatialpca  0.5688 ± 0.2663  0.9235 ± 0.8260  1.0000 ± 0.0000  2.3015 ± 3.0833
concord     0.4440 ± 0.0619  0.8581 ± 0.0131  1.0000 ± 0.0000  0.9585 ± 0.0396
fpca        0.8920 ± 0.0187  0.2674 ± 0.0138  1.0000 ± 0.0000  0.4082 ± 0.1706
